In [ ]:
#!/usr/bin/env python
"""
Topic Extraction Pipeline

Extracts the math topics and learning goals covered in each tutoring
session, using WizardLM-2 7B as a single-pass classifier over the full
concatenated session transcript.

Important: this runs ONCE PER SESSION, not once per segment. The resulting
topic list is then applied uniformly to every downstream dialogue segment
from that same session -- it's used by the "topic-context" prompting
strategy in 03_generate_llm_responses.py to tell the LLM what subject
matter this session is about.

Input:  all_sessions_segments.csv
        Raw per-turn log (must contain a transcriptID column and a text
        column with each turn's transcribed content).

Output: extract_topics_from_transcript_final.csv
        One row per session: transcriptID, combined_text, math_topics.

Supports resuming: if you re-run this script and the output file already
has some sessions in it, those sessions are skipped (so an interrupted run
doesn't waste re-processing already-completed sessions).
"""

import os
from typing import Optional

import ollama
import pandas as pd
from tqdm import tqdm

# ── Config -------------------------------------------------------------------
INPUT_PATH = "all_sessions_segments.csv"
OUTPUT_PATH = "extract_topics_from_transcript_final.csv"
TRANSCRIPT_ID_COL = "transcriptID"
TEXT_COL = "description"  # the column holding each turn's transcribed text


class TopicExtractionAgent:
    """
    Thin wrapper around one Ollama model call. Keeps the system prompt and
    the actual API call in one place so 03_generate_llm_responses.py's
    agent class (which does the same thing for tutor-utterance generation)
    can be compared side by side if you're checking prompting consistency.
    """

    def __init__(self, model: str = "wizardlm2:7b", temperature: float = 0.2,
                 top_k: int = 5, provider: Optional[str] = None):
        self.model = model
        self.options = {"temperature": temperature, "top_k": top_k}
        self.provider = "ollama"

    def _system_prompt(self) -> str:
        # This is the instruction that stays constant across every call --
        # it tells the model WHAT ROLE to play (a math analyst), not what
        # specific transcript to analyze (that comes in via chat()).
        return (
            "You are an expert math education analyst. "
            "Your job is to read tutoring session transcripts and identify the key math topics, "
            "concepts, and learning goals being covered. "
            "You always respond in a consistent, numbered list format. "
            "Be concise and specific. Use standard math terminology."
        )

    def chat(self, content: str) -> str:
        """Send one user message (the actual extraction prompt) and return the raw model reply."""
        messages = [
            {"role": "system", "content": self._system_prompt()},
            {"role": "user", "content": content},
        ]
        response = ollama.chat(model=self.model, messages=messages, options=self.options)
        return response["message"]["content"]


def make_extraction_prompt(transcript_text: str) -> str:
    """
    Build the actual per-session prompt: the full transcript, plus strict
    formatting instructions so we get a consistent, parseable output
    across all 50 sessions (rather than the model free-styling its answer
    format differently each time).
    """
    return (
        "Here is the transcript of a math tutoring session:\n\n"
        f"{transcript_text}\n\n"
        "Based on this transcript, identify the key math topics and learning goals covered.\n\n"
        "Instructions:\n"
        "- List only math topics and concepts (e.g. 'slope-intercept form', 'combining like terms')\n"
        "- Do NOT include off-topic conversation\n"
        "- Be specific, not vague (e.g. 'multiplying binomials' not just 'algebra')\n"
        "- Format your response exactly like this:\n\n"
        "MATH TOPICS COVERED:\n"
        "1. [topic]\n"
        "2. [topic]\n"
        "3. [topic]\n"
        "...\n\n"
        "MAIN LEARNING GOAL:\n"
        "[one sentence describing the overall goal of this session]"
    )


def extract_topics_for_all_sessions(input_path: str, output_path: str,
                                     transcript_col: str, text_col: str) -> pd.DataFrame:
    """
    Main extraction loop: one LLM call per session (not per turn/segment).
    """
    df = pd.read_csv(input_path)

    # Collapse every turn's text within a session into ONE big string per
    # session -- this is what gets fed to the model as "the transcript".
    sessions = df.groupby(transcript_col)[text_col].apply(lambda x: " ".join(x.astype(str))).reset_index()
    sessions = sessions.rename(columns={text_col: "combined_text"})

    agent = TopicExtractionAgent()

    # ── Resume logic ──────────────────────────────────────────────────────
    # If we've already run this before and got interrupted partway through,
    # don't waste API calls re-processing sessions we already have results
    # for. Just pick up where we left off.
    if os.path.exists(output_path):
        results_df = pd.read_csv(output_path)
        already_processed = set(results_df[transcript_col].tolist())
        print(f"Resuming: {len(already_processed)} sessions already processed.")
    else:
        results_df = pd.DataFrame(columns=[transcript_col, "combined_text", "math_topics"])
        already_processed = set()

    for _, row in tqdm(sessions.iterrows(), total=len(sessions), desc="Extracting math topics"):
        transcript_id = row[transcript_col]
        if transcript_id in already_processed:
            continue  # skip, already have this one from a previous run

        prompt = make_extraction_prompt(row["combined_text"])
        try:
            math_topics = agent.chat(prompt)
        except Exception as e:
            # Don't let one failed session kill the whole run -- log it and
            # move on with an empty topics string, which can be re-run later.
            print(f"Error processing transcript {transcript_id}: {e}")
            math_topics = ""

        results_df = pd.concat([results_df, pd.DataFrame([{
            transcript_col: transcript_id,
            "combined_text": row["combined_text"],
            "math_topics": math_topics,
        }])], ignore_index=True)

        # Save after EVERY session, not just at the end -- if this script
        # crashes or gets killed, we don't lose all the work already done.
        results_df.to_csv(output_path, index=False)

    return results_df


def main():
    result_df = extract_topics_for_all_sessions(
        input_path=INPUT_PATH,
        output_path=OUTPUT_PATH,
        transcript_col=TRANSCRIPT_ID_COL,
        text_col=TEXT_COL,
    )
    print(f"\nExtraction complete. Results saved to {OUTPUT_PATH}")
    print(result_df[[TRANSCRIPT_ID_COL, "math_topics"]].head())


if __name__ == "__main__":
    main()